# Quantum — Solve thật (không `--mock`) + Benchmark thật đầu tiên

Chạy `qshield-quantum solve` (không `--mock`) trên `action_effects.csv`/`pairwise_effects.csv`
thật từ `risk_effects.ipynb` — verify/consistency → exact (256 trạng thái) → QAOA (≥10 seed) →
chấm lại **true CVaR** qua `qshield_risk.evaluate()` (CLAUDE.md quy tắc 17, mới vừa hết blocked) →
benchmark exact/QAOA/classical.

## ⚠️ Notebook này BẮT BUỘC gọi qua subprocess — khác 3 notebook trước

Không được `from qshield_quantum.cli import solve; solve(...)` trực tiếp trong kernel. Hai lý do,
cả hai đã verify thật (không suy đoán):

1. **Segfault qiskit+pyarrow** (`docs/perf/2026-08-06-quantum-pipeline-implementation.md` §3.1):
   import `qshield_quantum` (qiskit) rồi ghi `.parquet` (pyarrow) trong CÙNG tiến trình sẽ crash.
2. **`os._exit(0)` giết kernel:** `qshield_quantum.cli.solve()` tự thoát tiến trình ngay khi chạy
   xong để né một bug treo khác của qiskit (`_fast_exit_if_standalone()`). Gọi trực tiếp trong
   kernel Jupyter (không phải subprocess, không phải pytest) sẽ **giết luôn kernel** — mất hết
   state, cell sau không chạy được nữa.

→ Notebook này gọi `qshield-quantum solve` qua `subprocess.run([sys.executable, "-c", ...])`,
đúng pattern `packages/pipeline/run.py` và test suite của `packages/quantum` đã dùng, rồi đọc lại
file JSON kết quả từ đĩa để hiển thị.

**Điều kiện tiên quyết:** đã chạy `notebooks/exploration/risk_effects.ipynb` — cần
`artifacts/dev/risk/{action_effects.csv,pairwise_effects.csv,baseline_risk.json}`.

In [1]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import yaml
from qshield_contracts.config import Config


def _find_project_root(marker: str = "CLAUDE.md") -> Path:
    p = Path.cwd().resolve()
    for candidate in (p, *p.parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(
        f"Không tìm thấy {marker} từ {p} trở lên — notebook phải nằm trong repo QSHIELD."
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)

CONFIG_PATH = PROJECT_ROOT / "configs" / "base.yaml"
cfg = Config.load(CONFIG_PATH)
cfg["artifacts"]["mode"]

'dev'

## Bước 0: Kiểm tra tiên quyết + config PROVISIONAL (giống hệt `risk_effects.ipynb`)

`qshield_risk.evaluate()` (được `solve()` gọi ở bước chấm true CVaR) cũng cần
`transaction_cost`/`weight_sum_tolerance` — dùng lại ĐÚNG số PROVISIONAL đã dùng ở
`risk_effects.ipynb` để nhất quán trong cùng một chuỗi kết quả.

In [2]:
required_risk_files = [
    PROJECT_ROOT / "artifacts" / "dev" / "risk" / "action_effects.csv",
    PROJECT_ROOT / "artifacts" / "dev" / "risk" / "pairwise_effects.csv",
    PROJECT_ROOT / "artifacts" / "dev" / "risk" / "baseline_risk.json",
    PROJECT_ROOT / "artifacts" / "dev" / "scenarios" / "scenario_manifest.json",
    PROJECT_ROOT / "artifacts" / "dev" / "scenarios" / "stress_scenarios.npz",
]
missing = [p for p in required_risk_files if not p.exists()]
if missing:
    raise RuntimeError(
        "Thiếu " + ", ".join(str(p) for p in missing) + " — chạy "
        "notebooks/exploration/risk_effects.ipynb trước."
    )
print(
    "OK — đủ action_effects/pairwise_effects/baseline_risk/scenario cube cho quantum."
)

OK — đủ action_effects/pairwise_effects/baseline_risk/scenario cube cho quantum.


In [3]:
from qshield_contracts.paths import ArtifactPaths

# PROVISIONAL — giống hệt risk_effects.ipynb (chưa phải số đã duyệt, TBD-002).
PROVISIONAL_TRANSACTION_COST = {
    "fee": 0.0015,
    "spread": 0.0010,
    "liquidity_penalty": 0.0005,
}
PROVISIONAL_WEIGHT_SUM_TOLERANCE = 1e-6

resolved_cfg = dict(cfg)
resolved_cfg["transaction_cost"] = PROVISIONAL_TRANSACTION_COST
resolved_cfg["weight_sum_tolerance"] = PROVISIONAL_WEIGHT_SUM_TOLERANCE

paths = ArtifactPaths(resolved_cfg, run_id=None)
paths.run_root.mkdir(parents=True, exist_ok=True)
resolved_config_path = paths.run_root / "_quantum_resolved_config.yaml"
resolved_config_path.write_text(
    yaml.safe_dump(resolved_cfg, allow_unicode=True), encoding="utf-8"
)
print(f"Config đã resolve → {resolved_config_path}")
print(
    "⚠️  NON_BASELINE_RUN — transaction_cost là placeholder, chưa được Phúc/Ngọc duyệt."
)

Config đã resolve → artifacts/dev/_quantum_resolved_config.yaml
⚠️  NON_BASELINE_RUN — transaction_cost là placeholder, chưa được Phúc/Ngọc duyệt.


## Bước 1: `qshield-quantum solve` (không `--mock`) qua subprocess

verify/consistency (quy tắc 15) → exact (256 trạng thái, quy tắc 16) → QAOA ≥10 seed (quy tắc 18:
không cherry-pick) → chấm true CVaR qua `qshield_risk.evaluate()` (quy tắc 17) → `benchmark.json`
(exact vs QAOA vs classical).

In [4]:
RUN_START = time.perf_counter()
result = subprocess.run(
    [
        sys.executable,
        "-c",
        "from qshield_quantum.cli import app; app()",
        "solve",
        "--config",
        str(resolved_config_path),
    ],
    capture_output=True,
    text=True,
    timeout=180,
    check=False,
)
total_seconds = time.perf_counter() - RUN_START
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)
    raise RuntimeError(f"qshield-quantum solve thoát với exit code {result.returncode}")
print(f"[timing] Solve (verify+exact+QAOA+benchmark+true CVaR): {total_seconds:.1f}s")

[quantum] OK — bitstring=01010100 → artifacts/dev/optimization/qaoa_result.json

[timing] Solve (verify+exact+QAOA+benchmark+true CVaR): 71.9s


## Bước 2: `qaoa_result.json` — nghiệm thắng + true CVaR

In [5]:
qaoa_result_path = (
    PROJECT_ROOT / "artifacts" / "dev" / "optimization" / "qaoa_result.json"
)
qaoa_result = json.loads(qaoa_result_path.read_text(encoding="utf-8"))
{
    "bitstring": qaoa_result["bitstring"],
    "chosen_actions": qaoa_result["chosen_actions"],
    "actual_solver": qaoa_result["actual_solver"],
    "optimality_gap": qaoa_result["optimality_gap"],
    "feasibility_rate": qaoa_result["feasibility_rate"],
    "true_cvar_before": qaoa_result["true_cvar_before"],
    "true_cvar_after": qaoa_result["true_cvar_after"],
    "cvar_improved": qaoa_result["true_cvar_after"] < qaoa_result["true_cvar_before"],
}

{'bitstring': '01010100',
 'chosen_actions': [1, 3, 5],
 'actual_solver': 'qaoa',
 'optimality_gap': -1.4634577273045616e-15,
 'feasibility_rate': 0.002635955810546875,
 'true_cvar_before': 0.1109732123107091,
 'true_cvar_after': 0.10157469115617396,
 'cvar_improved': True}

## Bước 3: `benchmark.json` — exact vs QAOA vs classical (thật, lần đầu tiên)

`qaoa_beats_classical=False` là kết quả HỢP LỆ, không phải lỗi — CLAUDE.md quy tắc 18 cấm diễn giải
có lợi cho QAOA khi nó thua. In nguyên trạng, không tô hồng.

In [6]:
benchmark_path = PROJECT_ROOT / "artifacts" / "dev" / "optimization" / "benchmark.json"
benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))
pd.Series(
    {
        "winning_seed": benchmark["winning_seed"],
        "winning_energy": benchmark["winning_energy"],
        "exact_best_feasible_energy": benchmark["exact_best_feasible_energy"],
        "classical_energy": benchmark["classical_energy"],
        "optimality_gap": benchmark["optimality_gap"],
        "classical_gap": benchmark["classical_gap"],
        "qaoa_beats_classical": benchmark["qaoa_beats_classical"],
        "n_seeds_feasible": f"{benchmark['n_seeds_feasible']}/{benchmark['n_seeds_total']}",
        "mean_feasibility_rate": benchmark["mean_feasibility_rate"],
    },
    name="benchmark",
)

winning_seed                         0
winning_energy               -0.009483
exact_best_feasible_energy   -0.009483
classical_energy             -0.009483
optimality_gap                    -0.0
classical_gap                      0.0
qaoa_beats_classical              True
n_seeds_feasible                 10/10
mean_feasibility_rate         0.002636
Name: benchmark, dtype: object

## Xong — checklist đầu ra

- `artifacts/dev/optimization/{qaoa_result.json,benchmark.json}`
- `artifacts/dev/{config.json,data_version.json,metrics.json,logs.txt}` — do `RunContext` ghi.
- `artifacts/dev/_quantum_resolved_config.yaml` — config thật đã dùng (kèm `transaction_cost`
  PROVISIONAL).

**⚠️ `NON_BASELINE_RUN`** — `transaction_cost` (qua `qshield_risk.evaluate`) và `penalty`
(`lambda_1/lambda_2/P`, tự suy ra bằng `suggest_penalty` vì `configs/quantum.yaml` còn `null`, TBD-006)
đều là số tạm, chưa Phúc/Ngọc duyệt. Đọc `logs.txt` trong `artifacts/dev/` để thấy đủ cảnh báo
PROVISIONAL đã ghi trong lúc chạy.

In [ ]:
# WORKFLOW_UPDATE downstream provisional — 8 candidates / 16 bits / 137 samples.
# Cell tự chứa và stream log trực tiếp; có thể dừng bằng nút Interrupt của notebook.
import os
import subprocess
import time


def find_project_root(marker: str = "CLAUDE.md") -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"Không tìm thấy {marker} từ {current}")


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
PROFILE = "configs/profiles/workflow_update.yaml"
OVERRIDE = "configs/provisional/workflow_update_downstream.yaml"
BASE = "configs/base.yaml"
commands = [
    (
        "1/3 Risk: dynamic top-min(N,10) + objective samples",
        [
            "uv",
            "run",
            "qshield-risk",
            "prepare-workflow",
            "--mock",
            "--config",
            BASE,
            "--profile",
            PROFILE,
            "--override",
            OVERRIDE,
        ],
    ),
    (
        "2/3 Quantum: surrogate + exact 2^16 + QAOA 10 seeds",
        [
            "uv",
            "run",
            "qshield-quantum",
            "workflow",
            "--config",
            BASE,
            "--profile",
            PROFILE,
            "--override",
            OVERRIDE,
        ],
    ),
    (
        "3/3 Risk: true-CVaR rerank + local polishing",
        [
            "uv",
            "run",
            "qshield-risk",
            "rerank-polish",
            "--mock",
            "--config",
            BASE,
            "--profile",
            PROFILE,
            "--override",
            OVERRIDE,
        ],
    ),
]

started_all = time.perf_counter()
for label, command in commands:
    print(f"\n{'=' * 80}\n{label}\n$ {' '.join(command)}", flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
    print(f"[{label}] exit={returncode}, elapsed={time.perf_counter() - started:.1f}s")
    if returncode != 0:
        raise RuntimeError(f"Stage thất bại: {label} (exit={returncode})")

print(
    f"\nDONE — NON_BASELINE_RUN sau {time.perf_counter() - started_all:.1f}s.\n"
    "Kết quả: artifacts/dev/risk/final_recommendation.json",
    flush=True,
)